# NPSC2002 Week 9 Workshop
## The Edge of the Solar System: New Horizons at Pluto

---

On 14 July 2015, NASA's New Horizons spacecraft completed the first close flyby of Pluto after a nine-year, 5-billion-kilometre journey. Travelling at 13.78 km/s relative to Pluto, it had no time to slow down or enter orbit. It had one chance to get the science right.

The primary imaging instrument was **LORRI** (Long Range Reconnaissance Imager): a 20 cm telescope with a 1024 x 1024 pixel detector. LORRI records light across all visible wavelengths in a single channel, so its images are greyscale. This simplicity is a strength: there is no colour calibration complexity, no channel alignment problem, and a very well-characterised point-spread function. LORRI produced the sharpest close-approach images ever taken of Pluto.

In this workshop you will retrieve real LORRI images from NASA's Planetary Data System and use them to re-derive two headline findings of the mission:

- **Pluto's true diameter**, which was uncertain by roughly 150 km before the flyby
- **Pluto's surface brightness variation**, which reveals a world of striking compositional contrasts

**Instructions:** run every code cell in order. Read the commentary beneath each output carefully. Write your answers in the cells marked *Your answer here*.

In [ ]:
!pip install astropy requests scipy pillow ipywidgets --quiet

In [ ]:
import requests
import io as _io
import base64
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
from matplotlib.ticker import AutoMinorLocator
from astropy.io import fits
from astropy.time import Time
import astropy.units as u
from scipy.ndimage import gaussian_filter, label, center_of_mass
from PIL import Image
import ipywidgets as widgets
from IPython.display import HTML, display

# ── Visual theme ─────────────────────────────────────────────────────────────
NAVY = '#0d1b2a'
GOLD = '#c9a84c'

plt.rcParams.update({
    'figure.facecolor':  NAVY,
    'axes.facecolor':    NAVY,
    'savefig.facecolor': NAVY,
    'text.color':        'white',
    'axes.labelcolor':   'white',
    'axes.edgecolor':    GOLD,
    'xtick.color':       'white',
    'ytick.color':       'white',
    'legend.facecolor':  '#1a2a3a',
    'legend.edgecolor':  GOLD,
    'legend.labelcolor': 'white',
    'grid.color':        '#2a3a4a',
    'grid.alpha':        0.5,
})

# ── Mission constants ─────────────────────────────────────────────────────────
OPUS         = 'https://opus.pds-rings.seti.org/api'
LORRI_IFOV   = 4.96e-6    # instrument plate scale: radians per pixel
CA_TIME      = Time('2015-07-14T11:49:00', format='isot', scale='utc')
CA_DIST_KM   = 12_472.0   # closest approach distance in km
NH_SPEED_KMS = 13.78      # NH approach speed relative to Pluto in km/s
PLUTO_D_KM   = 2_376.0    # accepted post-flyby diameter in km
PLUTO_D_PRE  = 2_300.0    # pre-flyby best estimate in km

print('Setup complete.')

In [ ]:
# =============================================================================
# Helper functions
# =============================================================================

def fetch_lorri(obs_id):
    """
    Download a calibrated LORRI FITS file from OPUS and return its pixel data.

    OPUS (Outer Planets Unified Search) hosts NASA planetary mission archives.
    Each observation has a unique obs_id string. We request the calibrated
    science file (_sci.fit) rather than the raw engineering file (_eng.fit):
    calibrated files have been flat-fielded, dark-subtracted, and converted to
    physical flux units (I/F: reflected intensity relative to incident sunlight,
    scaled by instrument response).

    Returns
    -------
    img : 2D float32 numpy array, shape (rows, cols)
    hdr : astropy FITS header with observation metadata
    """
    r = requests.get(f'{OPUS}/files/{obs_id}.json', timeout=15)
    cats = list(r.json()['data'].values())[0]
    fits_url = next(
        (url for cat, urls in cats.items()
         for url in urls if url.endswith('.fit') and 'calib' in cat),
        None
    )
    if not fits_url:
        raise ValueError(f'No calibrated FITS found for {obs_id}')
    resp = requests.get(fits_url, timeout=90)
    hdul = fits.open(_io.BytesIO(resp.content))
    img  = np.squeeze(hdul[0].data).astype(np.float32)
    hdr  = hdul[0].header
    hdul.close()
    return img, hdr


def nh_distance(obs_time_str):
    """
    Calculate the distance from New Horizons to Pluto at a given moment.

    Uses a linear flight model: distance = closest_approach_distance +
    (seconds before closest approach) x approach speed. Accurate to better
    than 1% over the final two weeks of approach.

    Parameters
    ----------
    obs_time_str : str  ISO timestamp, e.g. '2015-07-13T14:22:09.000'

    Returns
    -------
    float : distance in km
    """
    t    = Time(obs_time_str, format='isot', scale='utc')
    dt_s = (CA_TIME - t).to_value('s')
    return CA_DIST_KM + dt_s * NH_SPEED_KMS


def disc_display(img):
    """
    Prepare a LORRI image for display with a true-black background.

    Naive stretches (e.g. min-to-max or ZScale) often lift the sky background
    to a visible grey level because calibrated data contains small negative
    values from the calibration process. This hides faint surface detail and
    makes early-approach frames look washed out even when Pluto is a tiny dot.

    Steps:
    1. Mask hot pixels / cosmic rays (top 0.01% of values -> zero). These
       extreme outliers would otherwise dominate the stretch.
    2. Estimate the sky background from the image border (40-pixel margin)
       and subtract it, so empty sky maps to exactly zero (black).
    3. Clip remaining negative residuals to zero.
    4. Apply a square-root stretch: f_out = sqrt(f / f_max). This compresses
       bright regions (preventing Pluto from blowing out to white) while
       lifting faint surface detail into the visible range.

    Returns a float array in [0, 1] for plt.imshow(..., cmap='gray').
    """
    arr = img.copy()
    arr[arr > np.nanpercentile(arr, 99.99)] = 0.0            # Step 1
    border = np.concatenate([                                 # Step 2
        arr[:40, :].ravel(),  arr[-40:, :].ravel(),
        arr[:, :40].ravel(),  arr[:, -40:].ravel()
    ])
    bg   = np.nanmedian(border)
    arr  = np.clip(arr - bg, 0, None)                        # Step 3
    vmax = np.nanpercentile(arr, 99.5)
    return np.sqrt(np.clip(arr / (vmax + 1e-10), 0, 1))      # Step 4


def find_disc_centre(img, sigma=7):
    """
    Find the geometric centre of Pluto's disc in a LORRI image.

    WHY NOT USE THE BRIGHTEST PIXEL?
    Pluto's surface is not uniform: Tombaugh Regio (the 'heart') is
    significantly brighter than surrounding terrain. Using the brightness
    peak as the disc centre would bias every measurement towards the heart,
    systematically shifting the centroid away from the geometric centre.

    WHAT WE DO INSTEAD -- geometric centroid of the disc:
    1. Mask hot pixels (top 0.01% -> zero).
    2. Smooth with a Gaussian (sigma=7 px). At this scale, pixel-level
       brightness variations wash out; what remains is the disc-scale shape.
    3. Find the approximate disc location using the smoothed-image peak.
       This is still biased by the heart, but only by ~10-20 pixels, which
       is good enough to initialise the next step.
    4. Estimate the local sky background in a margin around the approximate
       peak, and threshold at background + 3 sigma to create a binary disc
       mask (True = disc, False = sky).
    5. Label connected regions in the mask. The disc is the connected region
       that contains the approximate peak.
    6. Compute the centre-of-mass of that labelled region on the smoothed
       image. Because the region was found from a heavily-smoothed image,
       the result reflects the geometric shape of the disc, not its surface
       brightness pattern.

    NOTE ON CHARON: In early frames (many days before closest approach),
    Charon may appear as a separate bright dot near Pluto. After Gaussian
    smoothing, Pluto is both larger and ~13x brighter than Charon, so the
    smoothed peak reliably lands on Pluto. The connected-region step then
    isolates only the Pluto disc, not Charon.

    Returns
    -------
    clean_img    : 2D float array with hot pixels zeroed
    centroid_row : int
    centroid_col : int
    """
    arr = img.copy()
    arr[arr > np.nanpercentile(arr, 99.99)] = 0.0              # Step 1
    smooth = gaussian_filter(np.clip(arr, 0, None), sigma)     # Step 2

    approx_r, approx_c = np.unravel_index(                    # Step 3
        np.argmax(smooth), smooth.shape)

    margin = min(300, smooth.shape[0] // 4, smooth.shape[1] // 4)  # Step 4
    r0 = max(0, approx_r - margin); r1 = min(smooth.shape[0], approx_r + margin)
    c0 = max(0, approx_c - margin); c1 = min(smooth.shape[1], approx_c + margin)
    region = smooth[r0:r1, c0:c1]
    bg     = np.nanmedian(region)
    bg_sub = region[region < np.nanpercentile(region, 70)]
    bg_std = np.nanstd(bg_sub) if len(bg_sub) > 0 else 1.0

    disc_mask      = smooth > bg + 3 * bg_std                 # Step 5
    labeled, n_lbl = label(disc_mask)

    if n_lbl > 0 and labeled[approx_r, approx_c] > 0:        # Step 6
        lbl    = labeled[approx_r, approx_c]
        cy, cx = center_of_mass(smooth, labeled, lbl)
        cy = int(np.clip(round(cy), 0, arr.shape[0] - 1))
        cx = int(np.clip(round(cx), 0, arr.shape[1] - 1))
        return arr, cy, cx

    print(f'  Warning: using smoothed peak as fallback ({approx_r}, {approx_c})')
    return arr, approx_r, approx_c


def measure_diameter(img, dist_km):
    """
    Measure Pluto's apparent diameter in pixels and convert to kilometres.

    HOW A BRIGHTNESS PROFILE WORKS:
    We extract one row of pixel values through the geometric disc centre
    (from find_disc_centre). Moving left to right across the image, this
    profile is flat at the sky background level, then rises steeply at
    Pluto's left limb, varies across the surface (reflecting albedo
    differences), and falls back to sky level at the right limb.

    EDGE DETECTION:
    We estimate the background level from the outermost 80 columns (sky only)
    and compute the noise standard deviation of that region. The detection
    threshold is background + 5 sigma. The leftmost and rightmost columns
    above this threshold are the disc edges.

    PIXEL TO KILOMETRE:
        km_per_pixel = distance_NH_Pluto (km) x LORRI_IFOV (rad/pixel)
        diameter_km  = diameter_pixels   x km_per_pixel

    LORRI_IFOV = 4.96e-6 rad/pixel is the angular width of one detector
    pixel, measured during pre-launch calibration. At distance d, one pixel
    subtends d x 4.96e-6 km on the sky.

    Returns
    -------
    dict with keys: diam_px, diam_km, km_per_px, pr, pc, profile,
                    bg, bg_std, thresh, left_edge, right_edge
    """
    arr, pr, pc = find_disc_centre(img)
    km_per_px   = dist_km * LORRI_IFOV

    profile    = arr[pr, :].astype(float)
    bg_pixels  = np.concatenate([profile[:80], profile[-80:]])
    bg         = np.median(bg_pixels)
    bg_std     = np.std(bg_pixels)
    thresh     = bg + 5 * bg_std

    cols_above = np.where(profile > thresh)[0]
    if len(cols_above) > 5:
        left_edge  = int(cols_above[0])
        right_edge = int(cols_above[-1])
        diam_px    = right_edge - left_edge
        diam_km    = diam_px * km_per_px
    else:
        left_edge = right_edge = diam_px = diam_km = None

    return dict(
        diam_px=diam_px, diam_km=diam_km, km_per_px=km_per_px,
        pr=pr, pc=pc, profile=profile,
        bg=bg, bg_std=bg_std, thresh=thresh,
        left_edge=left_edge, right_edge=right_edge
    )


print('Helper functions ready.')

---
## Part 1: Pluto growing larger

New Horizons photographed Pluto continuously during its final approach. The images below are real LORRI frames spaced on an **exponential schedule**: one frame every day or two at the start, shrinking to one per hour in the final approach. This mirrors the geometry: each passing day brought more new detail than the last.

Each image is shown at full LORRI frame size (1024 x 1024 pixels). In early frames Pluto is just a bright dot on a dark sky. You may also see **Charon**, Pluto's large moon, as a separate dot nearby. As New Horizons closes in, Pluto grows until it fills the detector. The last frames show just a patch of surface.

In [ ]:
# Generate target timestamps exponentially spaced in time-before-closest-approach.
# np.geomspace produces values equally spaced on a log scale, so the ratio
# between consecutive time intervals is constant -- frames cluster in the
# final hours where the geometry changes fastest.

N_FRAMES          = 30
START_BEFORE_CA_S = 15 * 24 * 3600   # 15 days before CA
END_BEFORE_CA_S   = 6.0 * 3600       # 6 hours before CA

times_before_ca = np.geomspace(START_BEFORE_CA_S, END_BEFORE_CA_S, N_FRAMES)
target_times    = [CA_TIME - dt * u.s for dt in times_before_ca]

print(f'Target schedule ({N_FRAMES} frames):')
for t, dt in zip(target_times, times_before_ca):
    dist = CA_DIST_KM + dt * NH_SPEED_KMS
    print(f'  {t.isot[:19]}  ({dt/3600:7.2f} h before CA,  {dist/1e6:.3f} million km)')

In [ ]:
# Query OPUS for the nearest available LORRI Pluto frame to each target time.
# Search window = 30% of the local frame spacing, capped at 12 hours.

approach = []
seen_ids = set()

for target_t, dt_s in zip(target_times, times_before_ca):
    half_win = min(dt_s * 0.3, 12 * 3600)
    w_start  = (target_t - half_win * u.s).isot[:19]
    w_end    = (target_t + half_win * u.s).isot[:19]

    try:
        r = requests.get(f'{OPUS}/data.json', params={
            'mission':    'New Horizons',
            'instrument': 'New Horizons LORRI',
            'target':     'Pluto',
            'cols':       'opusid,time1',
            'order':      'time1,opusid',
            'time1':      w_start,
            'time2':      w_end,
            'limit':      1, 'page': 1
        }, timeout=15)
        rows = r.json().get('page', [])
        if not rows:
            print(f'  {target_t.isot[:19]}: no data found')
            continue
        obs_id, obs_time = rows[0]
        if obs_id in seen_ids:
            continue
        seen_ids.add(obs_id)
        img, hdr = fetch_lorri(obs_id)
        dist_km  = nh_distance(obs_time)
        approach.append({'obs_id': obs_id, 'time': obs_time,
                         'dist_km': dist_km, 'img': img})
        print(f'  {obs_time[:19]}  {dist_km/1e6:.3f} Mkm  [{obs_id}]')
    except Exception as e:
        print(f'  {target_t.isot[:19]}: error ({e})')

print(f'\nTotal frames downloaded: {len(approach)}')

In [ ]:
# Display all approach frames at full LORRI resolution (1024x1024 pixels each).
# Background-subtracted sqrt stretch ensures sky is black and Pluto is
# visible from the earliest frames onward.
# Note the full timestamp (to the second) and distance in the subtitle of each panel.

n     = len(approach)
ncols = 6
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.4 * nrows))
axes = np.array(axes).flatten()

for ax, d in zip(axes, approach):
    ax.imshow(disc_display(d['img']), cmap='gray', vmin=0, vmax=1,
              origin='lower', aspect='equal', interpolation='nearest')
    ax.set_title(f"{d['time'][:19]}\n{d['dist_km']/1e6:.3f} Mkm",
                 color=GOLD, fontsize=7)
    ax.axis('off')

for ax in axes[n:]:
    ax.axis('off')

plt.suptitle('New Horizons LORRI: Pluto approach sequence  (full 1024x1024 frame, sqrt stretch)',
             color='white', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Build an animated GIF. Each full frame is resized to 300x300 px for the
# animation. Pluto grows from a faint dot to fill the detector; the last
# frames show individual terrain patches.

GIF_SIZE = 300
gif_pil  = []

for d in approach:
    stretched = disc_display(d['img'])                  # float [0,1], full frame
    f_u8      = (stretched * 255).astype(np.uint8)      # uint8 for PIL
    pil_img   = Image.fromarray(f_u8, mode='L')         # greyscale PIL image
    pil_img   = pil_img.resize((GIF_SIZE, GIF_SIZE), Image.NEAREST)
    gif_pil.append(pil_img.convert('RGB'))

buf = _io.BytesIO()
gif_pil[0].save(buf, format='GIF', save_all=True,
                append_images=gif_pil[1:], duration=600, loop=0)
buf.seek(0)
gif_b64 = base64.b64encode(buf.read()).decode()

display(HTML(
    f'<div style="background:{NAVY};padding:16px;text-align:center;">'
    f'<img src="data:image/gif;base64,{gif_b64}" '
    f'style="width:{GIF_SIZE}px;height:{GIF_SIZE}px;image-rendering:pixelated;'
    f'border:2px solid {GOLD};">'
    f'<p style="color:{GOLD};font-family:sans-serif;margin-top:8px;">'
    f'New Horizons LORRI: Pluto approach, July 1-14 2015 (full frame, looping)</p>'
    f'</div>'
))

**Questions**

1. Watch the GIF through several loops before answering. At roughly what distance (check the panel timestamps) does Pluto start to look like a world rather than a point of light? What surface features become visible once the disc is resolved?
2. Can you see Charon as a separate dot in the early frames? At what point does it disappear from the frame, and why?

*Your answer here.*

---
## Part 2: How big is Pluto?

Before New Horizons, Pluto's diameter was uncertain by roughly 150 km. Ground-based telescopes could not resolve the disc cleanly, and stellar occultation measurements were limited by how accurately Pluto's shadow path could be predicted. The best pre-flyby estimate was about 2,300 km. New Horizons settled it: the accepted post-flyby value is **2,376 km**, definitively making Pluto larger than Eris.

### How the measurement works

A **brightness profile** is a horizontal slice through the image: the pixel values along one row, read left to right. When that row passes through the geometric centre of Pluto's disc, the profile rises steeply from sky background at the left limb, varies across the surface (reflecting surface composition differences), and falls back to background at the right limb.

To locate the limbs we set a **detection threshold** at background + 5 standard deviations above the sky noise. The leftmost and rightmost columns above this threshold are the disc edges.

To convert pixels to kilometres:

$$D_{\text{km}} = D_{\text{px}} \times d_{\text{NH-Pluto}} \times \theta_{\text{LORRI}}$$

where $d$ is in km and $\theta_{\text{LORRI}} = 4.96 \times 10^{-6}$ rad/pixel.

If the method is correct, **every frame should give the same physical diameter** even though the apparent size in pixels changes dramatically across the approach. This consistency is the self-check.

In [ ]:
# Measure Pluto's diameter from every approach frame.
# Frames where the disc is smaller than MIN_DISC_PX pixels are skipped:
# below this size the threshold method cannot reliably separate disc from noise.

MIN_DISC_PX = 30
all_results = []

for d in approach:
    m = measure_diameter(d['img'], d['dist_km'])
    m.update({'time': d['time'], 'dist_km': d['dist_km'], 'obs_id': d['obs_id']})

    if m['diam_px'] and m['diam_px'] >= MIN_DISC_PX:
        err = abs(m['diam_km'] - PLUTO_D_KM) / PLUTO_D_KM * 100
        all_results.append(m)
        print(f"{d['time'][:19]}  {d['dist_km']/1e6:.3f} Mkm  "
              f"{m['km_per_px']:.2f} km/px  "
              f"{m['diam_px']:4d} px = {m['diam_km']:5.0f} km  (err {err:.1f}%)")
    else:
        tag = 'undetected' if not m['diam_px'] else f"{m['diam_px']} px (too small)"
        print(f"{d['time'][:19]}: skipped ({tag})")

### Your turn: choose which frames to include

Look at the approach panel and the diameter table above. Not every frame gives a reliable measurement:

- Very early frames may have a disc that is too small for accurate edge detection
- Very late frames may have Pluto's disc larger than the LORRI field of view, making the limbs invisible

In the cell below, set `MEASURE_FROM` and `MEASURE_TO` to the timestamps of the first and last frames you think show a clearly resolved disc with visible limbs on both sides. Use the format `'YYYY-MM-DDTHH:MM:SS'` — the full timestamps are printed in the table above.

In [ ]:
# ── EDIT THESE TWO VALUES ────────────────────────────────────────────────────
MEASURE_FROM = '2015-07-10T00:00:00'   # earliest frame to include
MEASURE_TO   = '2015-07-14T06:00:00'   # latest frame to include
# ────────────────────────────────────────────────────────────────────────────

results = [r for r in all_results
           if MEASURE_FROM <= r['time'][:19] <= MEASURE_TO]

print(f'Selected {len(results)} frames between {MEASURE_FROM} and {MEASURE_TO}:')
for r in results:
    print(f"  {r['time'][:19]}  {r['diam_px']} px = {r['diam_km']:.0f} km")

In [ ]:
# Show the brightness profile and edge detection for each selected frame.

if not results:
    print('No frames selected. Adjust MEASURE_FROM and MEASURE_TO above.')
else:
    n = len(results)
    fig, axes = plt.subplots(n, 1, figsize=(13, 3.2 * n))
    if n == 1:
        axes = [axes]

    for ax, r in zip(axes, results):
        x_km = (np.arange(len(r['profile'])) - r['pc']) * r['km_per_px']

        ax.plot(x_km, r['profile'],
                color='white', lw=0.6, alpha=0.3, label='Raw pixel values')
        ax.plot(x_km, gaussian_filter(r['profile'].copy(), 3),
                color='white', lw=1.8, label='Smoothed')
        ax.axhline(r['bg'],     color='#888', lw=1.0, ls=':',  label='Background')
        ax.axhline(r['thresh'], color=GOLD,   lw=1.0, ls='--',
                   label='Threshold (bg + 5 sigma)')

        if r['left_edge'] is not None:
            lx = (r['left_edge']  - r['pc']) * r['km_per_px']
            rx = (r['right_edge'] - r['pc']) * r['km_per_px']
            ax.axvline(lx, color='#6af', lw=1.5,
                       label=f"Limbs: {r['diam_km']:.0f} km apart")
            ax.axvline(rx, color='#6af', lw=1.5)
            ax.axvspan(lx, rx, alpha=0.07, color='white')

        ax.set_title(
            f"{r['time'][:19]}  |  {r['dist_km']/1e6:.3f} Mkm  |  "
            f"{r['km_per_px']:.2f} km/px  |  "
            f"{r['diam_px']} px = {r['diam_km']:.0f} km",
            color=GOLD, fontsize=9
        )
        ax.set_xlabel('Distance from disc centre (km)')
        ax.set_ylabel('Brightness (calibrated flux)')
        ax.set_xlim(-2500, 2500)
        ax.legend(fontsize=8, ncol=2)

    plt.tight_layout()
    plt.show()

In [ ]:
# Summary: apparent size in pixels (growing) vs derived physical diameter (should be constant).

if len(results) < 2:
    print('Select at least 2 frames above to see the summary plot.')
else:
    labels   = [r['time'][:10] for r in results]
    sizes_px = [r['diam_px']   for r in results]
    sizes_km = [r['diam_km']   for r in results]
    mean_km  = float(np.mean(sizes_km))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(range(len(labels)), sizes_px, 'o-', color=GOLD, lw=2, ms=9)
    ax1.set_xticks(range(len(labels)))
    ax1.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax1.set_title('Apparent disc size (pixels)', color=GOLD)
    ax1.set_ylabel('Diameter (px)')
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    ax1.grid(True)

    ax2.axhspan(PLUTO_D_PRE - 150, PLUTO_D_PRE + 150,
                alpha=0.18, color='cyan',
                label=f'Pre-NH estimate: {PLUTO_D_PRE:.0f} +/- 150 km')
    ax2.axhline(PLUTO_D_KM, color='white', lw=1.5, ls='--',
                label=f'Accepted: {PLUTO_D_KM:.0f} km')
    ax2.axhline(mean_km,    color=GOLD,    lw=1.0, ls=':',
                label=f'Your mean: {mean_km:.0f} km')
    ax2.plot(range(len(labels)), sizes_km, 'o', color=GOLD, ms=11, zorder=4,
             label='Individual measurements')
    ax2.set_xticks(range(len(labels)))
    ax2.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax2.set_title('Derived physical diameter', color=GOLD)
    ax2.set_ylabel('Diameter (km)')
    ax2.yaxis.set_minor_locator(AutoMinorLocator())
    ax2.legend(fontsize=9)
    ax2.grid(True)

    plt.suptitle('Pluto: apparent pixel size vs derived physical diameter',
                 color='white', fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f'Mean measured: {mean_km:.0f} km')
    print(f'Accepted:      {PLUTO_D_KM:.0f} km')
    print(f'Error:         {abs(mean_km - PLUTO_D_KM)/PLUTO_D_KM*100:.1f}%')
    print(f'Pre-NH:        {PLUTO_D_PRE:.0f} +/- 150 km')

**Questions**

1. Which frames did you exclude, and why? What visual criterion did you use to judge whether a frame gave a reliable measurement?
2. Are your derived diameter values consistent across the selected frames, even though the apparent pixel size changes dramatically? What does this consistency (or any inconsistency) tell you about the measurement method?

*Your answer here.*

---
## Part 3: Reading the surface

One of the most striking discoveries of the flyby was Pluto's surface: a world of sharp contrasts, with some of the brightest and darkest terrain in the outer solar system sitting side by side.

**Tombaugh Regio** (the 'heart') is the large bright equatorial region visible in the approach images. Its high albedo indicates freshly deposited nitrogen, methane, and carbon monoxide ice: volatile ices that sublime and re-condense seasonally, constantly renewing the surface.

**Cthulhu Macula** is the large dark equatorial belt to the west of the heart. It is coated in tholins: complex organic molecules that accumulate over billions of years when radiation processes nitrogen and methane. Tholins are very dark and reddish-brown.

A horizontal brightness profile across Pluto's disc captures this contrast directly.

The algorithm places a starting centroid automatically, but it may not land exactly in the geometric centre of the disc. Use the **interactive controls** below to:
- adjust the centroid row and column until the crosshair sits at the disc centre
- adjust the profile half-width to average over more rows (reduces noise)
- adjust the disc radius used to mark the limbs on the profile

In [ ]:
# Set up the frame and compute the initial centroid estimate.
# The interactive cell below will let you adjust from this starting point.

frame_s   = approach[-1]                           # last (closest) approach frame
arr_s, pr_init, pc_init = find_disc_centre(frame_s['img'])
dist_s    = frame_s['dist_km']
kpp_s     = dist_s * LORRI_IFOV

print(f"Frame:    {frame_s['obs_id']}")
print(f"Time:     {frame_s['time'][:19]}")
print(f"Distance: {dist_s/1e6:.3f} million km")
print(f"Scale:    {kpp_s:.2f} km per pixel")
print(f"Algorithm centroid: row={pr_init}, col={pc_init}")
print(f"Pluto radius at this distance: {PLUTO_D_KM/2/kpp_s:.0f} pixels")
print()
print('Run the next cell to open the interactive tool.')

In [ ]:
# Interactive surface brightness profile.

img_shape = arr_s.shape
r_px_init = int(round(PLUTO_D_KM / 2 / kpp_s))

@widgets.interact(
    centroid_row = widgets.IntSlider(
        value=pr_init,
        min=max(0,              pr_init - 400),
        max=min(img_shape[0]-1, pr_init + 400),
        step=1, description='Centre Y',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    ),
    centroid_col = widgets.IntSlider(
        value=pc_init,
        min=max(0,              pc_init - 400),
        max=min(img_shape[1]-1, pc_init + 400),
        step=1, description='Centre X',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    ),
    half_width = widgets.IntSlider(
        value=3, min=0, max=30, step=1,
        description='Profile half-width (rows)',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    ),
    disc_radius_km = widgets.IntSlider(
        value=int(PLUTO_D_KM / 2),
        min=600, max=1400, step=10,
        description='Disc radius (km)',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
)
def update_profile(centroid_row, centroid_col, half_width, disc_radius_km):

    fig, (ax_img, ax_prof) = plt.subplots(1, 2, figsize=(15, 6))

    # ── Left panel: full frame image ──────────────────────────────────────
    ax_img.imshow(disc_display(frame_s['img']), cmap='gray', vmin=0, vmax=1,
                  origin='lower', aspect='equal', interpolation='nearest')

    # Profile band (gold horizontal strip)
    r0_band = max(0, centroid_row - half_width)
    r1_band = min(img_shape[0] - 1, centroid_row + half_width)
    ax_img.axhspan(r0_band, r1_band, color=GOLD, alpha=0.25,
                   label=f'Profile band ({2*half_width+1} rows)')

    # Centroid crosshair
    ax_img.plot(centroid_col, centroid_row, '+',
                color=GOLD, ms=18, mew=2.5, label='Centroid')

    # Disc circle
    r_px = disc_radius_km / kpp_s
    circle = Circle((centroid_col, centroid_row), r_px,
                    fill=False, edgecolor=GOLD, lw=1.2, ls='--', alpha=0.8,
                    label=f'Disc edge (r = {disc_radius_km} km)')
    ax_img.add_patch(circle)

    ax_img.legend(fontsize=8, loc='lower right')
    ax_img.set_title(
        f"{frame_s['time'][:19]}  |  {dist_s/1e6:.3f} Mkm\n"
        f"centroid: row={centroid_row}, col={centroid_col}",
        color=GOLD, fontsize=9
    )
    ax_img.axis('off')

    # ── Right panel: brightness profile ───────────────────────────────────
    r0_avg = max(0, centroid_row - half_width)
    r1_avg = min(img_shape[0], centroid_row + half_width + 1)
    profile_avg = np.mean(arr_s[r0_avg:r1_avg, :], axis=0).astype(float)

    bg_pix  = np.concatenate([profile_avg[:80], profile_avg[-80:]])
    bg      = np.median(bg_pix)
    profile_clean = gaussian_filter(profile_avg - bg, sigma=2)

    x_km = (np.arange(len(profile_avg)) - centroid_col) * kpp_s

    disc_visible = (np.abs(x_km) < disc_radius_km) & (profile_clean > 0)
    ax_prof.fill_between(x_km, 0, profile_clean,
                         where=disc_visible, color=GOLD, alpha=0.15)
    ax_prof.plot(x_km, profile_avg - bg,
                 color='white', lw=0.6, alpha=0.3, label='Raw')
    ax_prof.plot(x_km, profile_clean,
                 color='white', lw=2,   label='Smoothed')
    ax_prof.axvline(-disc_radius_km, color='#888', lw=1.2, ls='--',
                    label=f'Disc edge (+/- {disc_radius_km} km)')
    ax_prof.axvline( disc_radius_km, color='#888', lw=1.2, ls='--')
    ax_prof.axvspan(-disc_radius_km, disc_radius_km,
                    alpha=0.05, color='white')
    ax_prof.axhline(0, color='#444', lw=0.8)

    # Mark brightest and darkest within the disc
    disc_mask = np.abs(x_km) < disc_radius_km
    if disc_mask.any():
        disc_x = x_km[disc_mask]
        disc_p = profile_clean[disc_mask]
        pk_x   = disc_x[np.argmax(disc_p)]
        dk_x   = disc_x[np.argmin(disc_p)]
        ax_prof.axvline(pk_x, color='#88f', lw=1.2, ls=':',
                        label=f'Brightest ({pk_x:.0f} km)')
        ax_prof.axvline(dk_x, color='#f88', lw=1.2, ls=':',
                        label=f'Darkest ({dk_x:.0f} km)')
        contrast = (disc_p.max() - disc_p.min()) / (abs(disc_p.min()) + disc_p.max() + 1e-10)
        print(f'Brightest point: {pk_x:.0f} km from centre  '
              f'| Darkest: {dk_x:.0f} km  '
              f'| Normalised contrast: {contrast:.2f}')

    ax_prof.set_xlim(-2000, 2000)
    ax_prof.set_xlabel('Distance from centroid (km)')
    ax_prof.set_ylabel('Brightness above background')
    ax_prof.set_title(
        f'Surface brightness profile  (profile band: {2*half_width+1} rows averaged)',
        color=GOLD, fontsize=9
    )
    ax_prof.legend(fontsize=8, ncol=2)
    ax_prof.yaxis.set_minor_locator(AutoMinorLocator())
    ax_prof.grid(True, which='major')

    plt.tight_layout()
    plt.show()

**Questions**

1. Use the controls above to select a cross-section of Pluto that has widely varying surface brightness.
2. Measure (by reading the graph on the right) the brightness ratio between brightest and darkest.
3. How was this a challenge for this mission? (hint: cameras on board New Horizons have expsoure time and gain that need to be set, they don't have automatic exposure control like on your phone camera).


*Your answer here.*